# EDA and Machine Learning — вступительное задание ИТМО «Искусственный интеллект»

Задача: бинарная классификация `relief_granted` (компания закрыла обращение с денежной или неденежной
компенсацией) для потока потребительских жалоб — фича «Приоритет обращения».

**Данные:** `data/complaints_train.csv` (~7.7 ГБ), `data/complaints_test.csv`, `data/sample_submission.csv`.
Отложенная часть — последние месяцы наблюдений, то есть будущее относительно обучающей.

**Инструменты:** DuckDB для операций по полному датасету (обучающий CSV не загружается целиком в pandas),
pandas / matplotlib для уже уменьшенных до безопасного размера выборок и визуализаций.

## Setup — общее окружение для всех заданий

Ячейка ниже выполняется один раз и определяет пути к данным, соединение DuckDB и SQL-выражения
для чтения исходных CSV. Все последующие задания переиспользуют их.

Важное замечание о чтении данных: текстовое поле `Consumer.Complaint.Narrative` содержит переносы строк
внутри закавыченных значений, поэтому файл читается как настоящий RFC-4180 CSV. Параллельный CSV-ридер
DuckDB на этом файле не работает (`Parallel CSV Reader currently does not support a full read on this file`),
поэтому используется `parallel=false`. Типы читаются как строки (`all_varchar=true`): данные сырые
(несколько форматов дат, мусорные значения), автоматическому выводу типов доверять нельзя — поля
разбираются осознанно там, где это нужно по заданию.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd


def _find_repo_root(start: Path) -> Path:
    """Ноутбук должен работать и из notebooks/, и из корня репозитория."""
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Не найден корень репозитория с каталогом data/")


ROOT = _find_repo_root(Path.cwd().resolve())
DATA = ROOT / "data"
TRAIN_CSV = DATA / "complaints_train.csv"
TEST_CSV = DATA / "complaints_test.csv"
SAMPLE_SUB_CSV = DATA / "sample_submission.csv"

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")  # чтобы виджет прогресса не попадал в вывод


def csv_src(path: Path) -> str:
    """SQL-выражение чтения сырого CSV: строгий разбор кавычек, без вывода типов."""
    return (
        f"read_csv('{path.as_posix()}', header=true, all_varchar=true, parallel=false)"
    )


TRAIN = csv_src(TRAIN_CSV)
TEST = csv_src(TEST_CSV)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

print(f"ROOT = {ROOT}")
for p in (TRAIN_CSV, TEST_CSV, SAMPLE_SUB_CSV):
    print(f"{p.name:>26}  exists={p.is_file()}  size={p.stat().st_size / 2**30:.2f} GiB")
print(f"duckdb {duckdb.__version__}, pandas {pd.__version__}")

ROOT = C:\work\eda-ml
      complaints_train.csv  exists=True  size=7.19 GiB
       complaints_test.csv  exists=True  size=0.39 GiB
     sample_submission.csv  exists=True  size=0.03 GiB
duckdb 1.5.5, pandas 3.0.5


## A1 — Размерность обучающего датасета

**Задание**

A1. Размерность
Сколько строк и колонок в обучающем датасете? Формат: строки; колонки. Обратите внимание: текстовое поле содержит переносы строк, наивное чтение файла даст неверный ответ.

Пример ответа: 123; 12

In [ ]:
# Корректный подсчёт: CSV разбирается с учётом кавычек, поэтому переносы строк
# внутри Consumer.Complaint.Narrative не создают лишних записей.
n_rows = con.sql(f"SELECT count(*) FROM {TRAIN}").fetchone()[0]
columns = list(con.sql(f"SELECT * FROM {TRAIN} LIMIT 0").columns)
n_cols = len(columns)

print(f"Строк (записей):  {n_rows:,}".replace(",", " "))
print(f"Колонок:          {n_cols}")
print()
for i, name in enumerate(columns, 1):
    print(f"{i:>3}. {name}")

In [ ]:
# Демонстрация ловушки из условия: наивный подсчёт физических строк файла
# (аналог `wc -l`) сильно завышает число записей, потому что многострочный
# текст жалобы разрывается на несколько строк файла.
naive_newlines = 0
with open(TRAIN_CSV, "rb") as f:
    while chunk := f.read(1 << 24):
        naive_newlines += chunk.count(b"\n")

naive_rows = naive_newlines - 1  # минус строка заголовка

print(f"Наивно (переводы строк минус заголовок): {naive_rows:,}".replace(",", " "))
print(f"Корректно (парсинг CSV с кавычками):     {n_rows:,}".replace(",", " "))
print(f"Завышение при наивном чтении:            {naive_rows / n_rows:.2f}x")

In [ ]:
# Независимая проверка другим парсером (pyarrow, newlines_in_values=True):
# число записей и колонок не должно зависеть от движка чтения.
import pyarrow.csv as pv

reader = pv.open_csv(
    TRAIN_CSV,
    read_options=pv.ReadOptions(block_size=1 << 26),
    parse_options=pv.ParseOptions(newlines_in_values=True),
)
arrow_rows, arrow_cols = 0, None
for batch in reader:
    arrow_rows += batch.num_rows
    arrow_cols = batch.num_columns

print(f"pyarrow: строк = {arrow_rows:,}".replace(",", " ") + f", колонок = {arrow_cols}")
print(f"duckdb : строк = {n_rows:,}".replace(",", " ") + f", колонок = {n_cols}")
assert (arrow_rows, arrow_cols) == (n_rows, n_cols), "Парсеры разошлись в размерности"
print("\nОтвет A1 →", f"{n_rows}; {n_cols}")

**Ответ:** 13367673; 18

**Вывод:** обучающий датасет содержит 13 367 673 записи и 18 колонок (17 признаков плюс таргет
`relief_granted`). Наивный подсчёт физических строк файла даёт 23 307 172 — завышение в ~1.74 раза,
потому что поле `Consumer.Complaint.Narrative` содержит переносы строк внутри закавыченных значений.
Одна запись данных ≠ одна строка файла, поэтому весь дальнейший анализ идёт через CSV-парсер,
учитывающий кавычки. Дополнительно зафиксировано: параллельный CSV-ридер DuckDB на этом файле
неприменим, чтение выполняется в режиме `parallel=false`; результат подтверждён вторым независимым
парсером (pyarrow).